# BTP step S1c on Kaggle: SAE validation and E0 (gemma-2-2b-it)

Runs `scripts/s1_sae.sh gemma-2-2b-it 2` on **GPU T4 x2**, meant for a background "Save & Run All (Commit)" run:
1,000 greedy GSM8K-train traces, SAE validation on all 26 layers (first 200 traces), then the E0 sparsity check.
The HF model generates on `cuda:0`; for SAE validation and E0 the TransformerLens copy (float32) is split over both GPUs.

Pre-registration: `prereg/E0.md` (committed before any S1c data existed).
**Before running:** Accelerator GPU T4 x2, Internet On, input `btp-code` (version with `n_devices` in `analysis/sae_validation.py`), secret `HF_TOKEN` attached.
Any failing step raises, so a failed commit run stops at that cell; `/kaggle/working/s1c.log` is kept either way.

In [ ]:
!nvidia-smi
!python --version
import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 2, 'Need 2 GPUs: set Accelerator to GPU T4 x2'
!df -h /kaggle/working /root /tmp

In [ ]:
# Find the code under /kaggle/input: Kaggle may keep btp_code.zip or auto-extract it into a folder.
import glob, os, shutil, zipfile
DST = '/kaggle/working/Reasoning-Failure-Diagnosis'
shutil.rmtree(DST, ignore_errors=True)
zips = glob.glob('/kaggle/input/**/btp_code.zip', recursive=True)
reqs = [p for p in glob.glob('/kaggle/input/**/requirements.txt', recursive=True)
        if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
if zips:
    print('zip:', zips[0])
    tmp = '/kaggle/working/_unzip'
    shutil.rmtree(tmp, ignore_errors=True)
    zipfile.ZipFile(zips[0]).extractall(tmp)
    inner = [os.path.dirname(p) for p in glob.glob(tmp + '/**/requirements.txt', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'scripts', 's1_equivalence.sh'))]
    assert inner, 'no project root inside the zip'
    shutil.copytree(min(inner, key=len), DST)
    shutil.rmtree(tmp)
elif reqs:
    src = os.path.dirname(min(reqs, key=len))
    print('folder:', src)
    shutil.copytree(src, DST)
else:
    raise FileNotFoundError('btp_code not found under /kaggle/input; add the btp-code dataset as input')
assert 'numpy>=2.0' in open(f'{DST}/requirements.txt').read(), 'stale requirements.txt'
print(sorted(os.listdir(DST)))
assert 'hf_reference' in open(f'{DST}/harness/equivalence.py').read(), 'old btp-code version: update the dataset input'
assert 'n_devices' in open(f'{DST}/analysis/sae_validation.py').read(), 'old btp-code version (no S1c 2-GPU patch): update the dataset input'
assert 'Date committed:** 2026-11-11' in open(f'{DST}/prereg/E0.md').read(), 'E0 pre-registration not committed'

In [ ]:
# requirements.txt needs Python 3.13 (TransformerLens 2.x caps numpy<2 on <=3.12), so build a 3.13 venv with uv.
!pip -q install uv
!uv venv --seed --python 3.13 /kaggle/working/venv
!uv pip install --python /kaggle/working/venv/bin/python -r /kaggle/working/Reasoning-Failure-Diagnosis/requirements.txt
os.environ['PATH'] = '/kaggle/working/venv/bin:' + os.environ['PATH']
os.environ['VIRTUAL_ENV'] = '/kaggle/working/venv'
!which python && python -c "import torch, transformer_lens, numpy; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), numpy.__version__)"

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')   # never printed
print('HF_TOKEN set:', bool(os.environ['HF_TOKEN']))

In [ ]:
!cd /kaggle/working/Reasoning-Failure-Diagnosis && python -m pytest -q
assert _exit_code == 0, 'pytest failed'

In [ ]:
# Smoke test of every S1c stage on 2 GSM8K-train problems (seed 99, not the pre-registered sample),
# written to /tmp/smoke and never reported: catches 2-GPU/SAE bugs in minutes instead of after hours.
SMOKE = r"""set -euo pipefail
cd /kaggle/working/Reasoning-Failure-Diagnosis
S=/tmp/smoke; rm -rf $S; mkdir -p $S
python -m harness.datasets --source gsm8k --split train --n 2 --seed 99 --out $S/p.jsonl
python -c "from harness.generate import run; from harness.prompts import PromptSpec; run('gemma-2-2b-it', '$S/p.jsonl', '$S/t.jsonl', specs=[PromptSpec('marked', 0), PromptSpec('terse', 0)])"
python -m harness.score --traces $S/t.jsonl --out $S/ts.jsonl
python -m analysis.sae_validation --model gemma-2-2b-it --traces $S/ts.jsonl --layers 0 13 25 --max-traces 2 --n-devices 2 --out $S/v.json
python -m analysis.sparsity_e0 collect --model gemma-2-2b-it --traces $S/ts.jsonl --sae-report $S/v.json --max-traces 4 --layers-per-band 1 --n-devices 2 --out $S/e0.csv.gz
python -m analysis.sparsity_e0 analyse --tokens $S/e0.csv.gz --n-layers 26 --resamples 200 --out $S/e0.json
"""
open('/tmp/smoke.sh', 'w').write(SMOKE)
!bash /tmp/smoke.sh
assert _exit_code == 0, 'smoke test failed'
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv
!df -h /root /kaggle/working

In [ ]:
# S1c run (several hours). The log keeps everything, including a failure's traceback.
!cd /kaggle/working/Reasoning-Failure-Diagnosis && bash -c 'set -o pipefail; bash scripts/s1_sae.sh gemma-2-2b-it 2 2>&1 | tee /kaggle/working/s1c.log'
assert _exit_code == 0, 's1_sae.sh failed: see /kaggle/working/s1c.log'

In [ ]:
# Collect outputs into /kaggle/working/out and summarise.
import glob, json, os, shutil
P = '/kaggle/working/Reasoning-Failure-Diagnosis'
OUT = '/kaggle/working/out'
os.makedirs(OUT + '/reports_m0', exist_ok=True); os.makedirs(OUT + '/generated', exist_ok=True)
for f in glob.glob(P + '/reports/m0/*'): shutil.copy(f, OUT + '/reports_m0/')
for f in glob.glob(P + '/data/generated/gemma-2-2b-it/*.jsonl') + glob.glob(P + '/data/generated/gemma-2-2b-it/e0_tokens.csv.gz'):
    shutil.copy(f, OUT + '/generated/')
shutil.copy(P + '/data/m0_gsm8k_train500.jsonl', OUT + '/generated/')
shutil.copy('/kaggle/working/s1c.log', OUT + '/')
for root, _, fs in os.walk(OUT):
    for f in sorted(fs): print(os.path.getsize(os.path.join(root, f)), os.path.join(root, f))
v = json.load(open(OUT + '/reports_m0/sae_validation_gemma-2-2b-it.json'))
for r in v['layers']: print(f"layer {r['layer']:2d} FVU {r['fvu']:.4f} dCE {r['delta_ce']:.4f} {'PASS' if r['pass'] else 'FAIL'}")
print('passing layers:', v['passing_layers'])
print(json.dumps(json.load(open(OUT + '/reports_m0/e0_gemma-2-2b-it.json')), indent=1))
shutil.make_archive('/kaggle/working/s1c_out', 'zip', OUT)